## Movie Recomandiation System

In [ ]:
## Load the data
import pandas as pd
import numpy as np
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")
movies.head()

In [ ]:
credits.head(1).values

In [ ]:
## merege both data
movies = movies.merge(credits, on='title')

###  konse column import hai(mtlb data me kya kya impt hai ) 

-------- ye attribute impt hai   
- ```genres```
- ```id```
- ```keyword```
- ```Title```
-  ```overview```
- ```cast ```
- ```crew```

In [ ]:
## main data that in work in future
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]
movies.head()

In [ ]:
movies.isnull().sum()

In [ ]:
movies.dropna(inplace=True)

In [ ]:
movies.duplicated().sum()

In [ ]:
movies.iloc[0].genres

##  ```List ek string hai to us string ko list me convert krne ke liye import  ast use kiya hai uske baad usme se name liiye hai or ek L list me daal diye hai``` 

In [ ]:

import ast
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
     L.append(i['name'])
    return L


In [ ]:
movies['genres'] = movies['genres'].apply(convert)

In [ ]:
movies['genres']

## ```convert keyword attributes into list and take only name  as save as in keywords attribte```

In [ ]:
movies['keywords'] = movies['keywords'].apply(convert)

## ```convert cast attributes into list and take only 3 dictonary```

In [ ]:
import ast

def convert3(obj):
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
            L.append(i['name'])
            counter += 1
        else:
            break
    return L

In [ ]:
movies['cast']= movies['cast'].apply(convert3)

In [ ]:
movies['cast']

In [ ]:
movies.head()

In [ ]:
movies['crew'][0]

###  CLEAN ```CREW``` ATTRIBUTES 

In [ ]:
import ast

def fetch_director(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L
movies['crew'] = movies['crew'].apply(fetch_director)


In [ ]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

- ##  ``` Remove spaces from all  atributes ```

In [ ]:
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])


In [ ]:
movies.head()

-  ### ```Add A Column "TAG" whihc is the combination of [overview,genres keywords cast crew] in form of string```

In [ ]:
movies['tags'] =  movies['overview']+movies['genres']+movies['keywords']+movies["cast"]+movies['crew']

```Create a new DataFrame using of existing data frame(movies)```

In [ ]:
new_df = movies[['movie_id','title','tags']]


In [ ]:
new_df

In [ ]:
new_df['tags']  = new_df['tags'].apply(lambda x:" ".join(x))

In [ ]:
new_df.head()

In [ ]:
new_df['tags'][0]

- ```convert in into lower case```

In [ ]:
new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())

In [ ]:
(new_df['tags'][0])

## Apply Stemming 

## 📌 Stemming ()

### 🔹 Definition

**Stemming** is a text preprocessing technique used in Natural Language Processing (NLP) to reduce words to their root or base form by removing suffixes and prefixes.

---

### 🔹 Example

* running → run
* runs → run
* studies → studi

> Note: The resulting word may not always be a valid English word.

---

### 🔹 Purpose

* Reduces different forms of a word to a single form
* Helps in decreasing data size
* Improves model performance in NLP tasks

---

### 🔹 Output

```
run  
run  
studi  
```
---

### 🔹 One-Line Summary

👉 **Stemming reduces words to their root form by removing suffixes, without considering meaning.**


In [ ]:
import nltk
from nltk.stem.porter import PorterStemmer
ps  = PorterStemmer()

In [ ]:
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [ ]:
new_df['tags']  = new_df['tags'].apply(stem)

## ```Time to  Text  Vectorization```

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv =  CountVectorizer(max_features=5000, stop_words='english')

In [ ]:
vecotor = cv.fit_transform(new_df['tags']).toarray()                #convert text into a vecotor

In [ ]:
vecotor[0]

In [ ]:
list(cv.get_feature_names_out())

```compare distance in form of cosine of all vectors data ```

In [175]:
from sklearn.metrics.pairwise import cosine_similarity

In [177]:
cosine_similarity(vecotor).shape

(4806, 4806)

In [178]:
similarity = cosine_similarity(vecotor)

In [192]:
sorted(list(enumerate(similarity[0])),reverse=True,key=lambda x:x[1])[1:6]

[(1214, np.float64(0.28676966733820225)),
 (2405, np.float64(0.26901379342448517)),
 (3728, np.float64(0.2605130246476754)),
 (507, np.float64(0.255608593705383)),
 (539, np.float64(0.25038669783359574))]

```Create a function to recomand top 5 movie  in which distance of movie (cosine distance) is closer```

In [199]:
def recomand(movie):
    movie_index = new_df[new_df['title']==movie].index[0]
    distances = similarity[movie_index]
    movie_list = sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]
    for i in movie_list:
        print(new_df.iloc[i[0]].title)
     
    return

In [200]:
recomand('Avatar')

Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.


In [201]:
import pickle

In [202]:
pickle.dump(new_df,open("movies.pkl",'wb'))

In [203]:
pickle.dump(similarity,open('similarity.pkl','wb'))